In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler,MinMaxScaler,RobustScaler
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

In [3]:
url='https://raw.githubusercontent.com/sachin365123/CSV-files-for-Data-Science-and-Machine-Learning/main/Loan%20Approval%20Prediction.csv'
df=pd.read_csv(url)

In [4]:
df

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


In [5]:
X=df.drop(columns=['Loan_ID','Loan_Status'])
y=df['Loan_Status']
X

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban
...,...,...,...,...,...,...,...,...,...,...,...
609,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural
610,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural
611,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban
612,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban


In [6]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             601 non-null    object 
 1   Married            611 non-null    object 
 2   Dependents         599 non-null    object 
 3   Education          614 non-null    object 
 4   Self_Employed      582 non-null    object 
 5   ApplicantIncome    614 non-null    int64  
 6   CoapplicantIncome  614 non-null    float64
 7   LoanAmount         592 non-null    float64
 8   Loan_Amount_Term   600 non-null    float64
 9   Credit_History     564 non-null    float64
 10  Property_Area      614 non-null    object 
dtypes: float64(4), int64(1), object(6)
memory usage: 52.9+ KB


In [7]:
X.isnull().sum()

Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
dtype: int64

In [8]:
#null_col=[var for var in df.columns if df[var].isnull().mean()<0.05 and df[var].isnull().mean()>0]
#df.dropna(subset=null_col,inplace=True)

In [9]:
num_cols = ['ApplicantIncome','CoapplicantIncome','LoanAmount',
            'Loan_Amount_Term','Credit_History']

ohe_cols = ['Gender','Married','Self_Employed']

ord_cols = ['Dependents','Education','Property_Area']

In [19]:
'''tf1=ColumnTransformer([
    ('tf1',SimpleImputer(),['Credit_History']),
    ('tf2',SimpleImputer(strategy='most_frequent'),['Self_Employed'])
],remainder='passthrough',verbose_feature_names_out=False)
tf1.set_output(transform='pandas')
X_imputed=tf1.fit_transform(X)
X_imputed'''

"tf1=ColumnTransformer([\n    ('tf1',SimpleImputer(),['Credit_History']),\n    ('tf2',SimpleImputer(strategy='most_frequent'),['Self_Employed'])\n],remainder='passthrough',verbose_feature_names_out=False)\ntf1.set_output(transform='pandas')\nX_imputed=tf1.fit_transform(X)\nX_imputed"

In [20]:
num_pipeline = Pipeline([
    ('imputer',SimpleImputer()),
    ('scaler', StandardScaler())
])

In [21]:
cat_ord_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[
        ['0','1','2','3+'],
        ['Not Graduate','Graduate'],
        ['Rural','Semiurban','Urban']
    ]))
])

In [22]:
cat_ohe_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first',
                          handle_unknown='ignore',
                          sparse_output=False))
])

In [23]:
preprocess = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('ohe', cat_ohe_pipeline, ohe_cols),
    ('ord', cat_ord_pipeline, ord_cols)
], remainder='passthrough',verbose_feature_names_out=False)
preprocess.set_output(transform='pandas')
X_transformed = preprocess.fit_transform(X)
X_transformed

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Gender_Male,Married_Yes,Self_Employed_Yes,Dependents,Education,Property_Area
0,0.072991,-0.554487,0.000000,0.279851,0.451640,1.0,0.0,0.0,0.0,1.0,2.0
1,-0.134412,-0.038732,-0.219273,0.279851,0.451640,1.0,1.0,0.0,1.0,1.0,0.0
2,-0.393747,-0.554487,-0.957641,0.279851,0.451640,1.0,1.0,1.0,0.0,1.0,2.0
3,-0.462062,0.251980,-0.314547,0.279851,0.451640,1.0,1.0,0.0,0.0,0.0,2.0
4,0.097728,-0.554487,-0.064454,0.279851,0.451640,1.0,0.0,0.0,0.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...
609,-0.410130,-0.554487,-0.898095,0.279851,0.451640,0.0,0.0,0.0,0.0,1.0,0.0
610,-0.212557,-0.554487,-1.267279,-2.518655,0.451640,1.0,1.0,0.0,3.0,1.0,0.0
611,0.437174,-0.472404,1.269371,0.279851,0.451640,1.0,1.0,0.0,1.0,1.0,2.0
612,0.357064,-0.554487,0.483367,0.279851,0.451640,1.0,1.0,0.0,2.0,1.0,2.0


In [24]:
le=LabelEncoder()
y_transformed=le.fit_transform(y)

In [25]:
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression())
])


In [26]:
params={
    'preprocess__num__imputer__strategy':['mean','median'],
    'preprocess__num__imputer__add_indicator':[True,False],
    'preprocess__ohe__imputer__add_indicator':[True,False],
    'preprocess__ohe__imputer__add_indicator':[True,False],
    'model__C':[0.1,0.01,0.001,0.5,1.0,2.0,5.0]
}

In [27]:
grid=GridSearchCV(pipe,param_grid=params,scoring='accuracy',cv=10, n_jobs=-1)

In [28]:
X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2)

In [29]:
grid.fit(X_train, y_train)

,estimator,Pipeline(step...egression())])
,param_grid,"{'model__C': [0.1, 0.01, ...], 'preprocess__num__imputer__add_indicator': [True, False], 'preprocess__num__imputer__strategy': ['mean', 'median'], 'preprocess__ohe__imputer__add_indicator': [True, False]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('ohe', ...), ...]"


In [30]:
grid.best_params_

{'model__C': 0.1,
 'preprocess__num__imputer__add_indicator': True,
 'preprocess__num__imputer__strategy': 'mean',
 'preprocess__ohe__imputer__add_indicator': True}

In [32]:
grid.best_score_

np.float64(0.8106122448979592)

In [31]:
cross_val_score(grid,X_train,y_train,scoring='accuracy',cv=5,n_jobs=-1).mean()

np.float64(0.8105751391465678)

In [33]:
y_pred=grid.predict(X_test)

In [34]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.8048780487804879